# Adaptive Retrieval-Augmented Generation (RAG)

## Overview

Traditional RAG uses the **same retrieval strategy** for every question. But different questions need different approaches:

| Query Type | Example | Strategy |
|---|---|---|
| **Factual** | "What is the distance to the Sun?" | Enhance the query for precision, then rank by relevance |
| **Analytical** | "How does distance from the Sun affect climate?" | Break into sub-questions, retrieve broadly, select diverse results |
| **Opinion** | "What are theories about the origin of life?" | Find multiple viewpoints, ensure diversity |
| **Contextual** | "How does Earth's position affect habitability?" | Incorporate user context, personalize retrieval |

**Adaptive RAG** first classifies the query, then picks the right strategy.

## Models Used

- **LLM**: `gemma3:4b` via Ollama (local)
- **Embeddings**: `mxbai-embed-large:335m` via Ollama (local)

<div style="text-align: center;">

<img src="./images/adaptive_retrieval.svg" alt="adaptive retrieval" style="width:100%; height:auto;">
</div>

---
## Step 0: Import Packages

In [1]:
from langchain_community.vectorstores import FAISS
from langchain_ollama.embeddings import OllamaEmbeddings
from langchain_text_splitters import CharacterTextSplitter
from langchain_core.prompts import PromptTemplate
from langchain_ollama import ChatOllama

---
## Step 1: Set Up the LLMs and Embedding Model

We use two LLMs:
- **Small LLM** (`gemma3:4b`) — fast, used for classification and ranking.
- **Large LLM** (`gemma3:12b`) — more capable, used for sub-query generation and final answers.

In [2]:
llm = ChatOllama(temperature=0, model="gemma3:12b", max_tokens=4000)
embedding_model = OllamaEmbeddings(model="mxbai-embed-large:335m")

print("Models ready")

Models ready


---
## Step 2: Prepare Documents and Create the Vector Store

We start with a simple example text. In a real system, this would be your document corpus.

In [3]:
texts = [
    "The Earth is the third planet from the Sun and the only astronomical object known to harbor life."
]

text_splitter = CharacterTextSplitter(chunk_size=800, chunk_overlap=0)
documents = text_splitter.create_documents(texts)
vector_store = FAISS.from_documents(documents, embedding_model)

print(f"Created vector store with {len(documents)} document(s)")
print(f"Document: {documents[0].page_content}")

Created vector store with 1 document(s)
Document: The Earth is the third planet from the Sun and the only astronomical object known to harbor life.


---
## Step 3: Define the JSON Schemas for Structured LLM Output

We need the LLM to return structured data (not free text) at several points. These schemas tell the LLM exactly what format to use.

In [4]:
# Schema for query classification
category_schema = {
    "title": "CategoryOptions",
    "type": "object",
    "properties": {
        "category": {
            "type": "string",
            "description": "The category of the query. Options: Factual, Analytical, Opinion, or Contextual"
        }
    },
    "required": ["category"]
}

# Schema for relevance scoring
relevance_score_schema = {
    "title": "RelevantScore",
    "type": "object",
    "properties": {
        "score": {
            "type": "number",
            "description": "The relevance score of the document to the query (1-10)"
        }
    },
    "required": ["score"]
}

# Schema for selecting document indices
selected_indices_schema = {
    "title": "SelectedIndices",
    "type": "object",
    "properties": {
        "indices": {
            "type": "array",
            "items": {"type": "integer"},
            "description": "Indices of selected documents"
        }
    },
    "required": ["indices"]
}

# Schema for sub-queries
sub_queries_schema = {
    "title": "SubQueries",
    "type": "object",
    "properties": {
        "sub_queries": {
            "type": "array",
            "items": {"type": "string"},
            "description": "List of sub-queries for comprehensive analysis"
        }
    },
    "required": ["sub_queries"]
}

print("Schemas defined")

Schemas defined


---
## Step 4: Define the Answer Generation Prompt

After retrieval, we always use the same prompt to generate the final answer from the retrieved context.

In [5]:
answer_prompt = PromptTemplate(
    input_variables=["context", "question"],
    template=(
        "Use the following pieces of context to answer the question at the end. "
        "If you don't know the answer, just say that you don't know, don't try to make up an answer.\n\n"
        "{context}\n\n"
        "Question: {question}\n"
        "Answer:"
    )
)
answer_chain = answer_prompt | llm

print("Answer generation chain ready")

Answer generation chain ready


---
---
# Demo 1: Factual Query

**Strategy:** Enhance the query for precision → Retrieve → Rank by relevance → Answer

---
## Step 5a: Classify the Query

In [6]:
query = "What is the distance between the Earth and the Sun?"

# Classify the query
classify_prompt = PromptTemplate(
    input_variables=["query"],
    template="Classify the following query into one of these categories: Factual, Analytical, Opinion, or Contextual.\nQuery: {query}\nCategory:"
)
classify_chain = classify_prompt | llm.with_structured_output(category_schema)

category = classify_chain.invoke(query)["category"]
print(f"Query: {query}")
print(f"Category: {category}")

Query: What is the distance between the Earth and the Sun?
Category: Factual


---
## Step 5b: Factual Strategy — Enhance the Query

For factual queries, we first ask the LLM to rewrite the query for better precision.

In [7]:
enhance_prompt = PromptTemplate(
    input_variables=["query"],
    template="Enhance this factual query for better information retrieval: {query}"
)
enhance_chain = enhance_prompt | llm

enhanced_query = enhance_chain.invoke(query).content
print(f"Original query: {query}")
print(f"\nEnhanced query: {enhanced_query}")

Original query: What is the distance between the Earth and the Sun?

Enhanced query: Okay, here are several enhanced versions of the query "What is the distance between the Earth and the Sun?" designed to yield better information retrieval, along with explanations of *why* each enhancement is helpful.  I've categorized them by increasing complexity and specificity.

**1. Basic Enhancement (Adding Context & Avoiding Ambiguity):**

* **Query:** "Average distance between Earth and Sun"
* **Why it's better:**  The distance isn't constant.  Adding "average" clarifies that you're looking for a typical value, not a specific moment in time.  This avoids results about the perihelion (closest point) or aphelion (farthest point).

**2. Adding Units & Specificity:**

* **Query:** "Average distance between Earth and Sun in kilometers"  OR "Average distance between Earth and Sun in astronomical units"
* **Why it's better:**
    * **Units:**  Specifying "kilometers" or "astronomical units" (AU) ensur

---
## Step 5c: Factual Strategy — Retrieve and Rank

We retrieve extra documents (2x), then ask the LLM to score each one for relevance (1–10). We keep only the top-k.

In [8]:
k = 4

# Retrieve 2x documents using the enhanced query
docs = vector_store.similarity_search(enhanced_query, k=k * 2)
print(f"Retrieved {len(docs)} documents")

# Rank each document by relevance using the LLM
ranking_prompt = PromptTemplate(
    input_variables=["query", "doc"],
    template="On a scale of 1-10, how relevant is this document to the query: '{query}'?\nDocument: {doc}\nRelevance score:"
)
ranking_chain = ranking_prompt | llm.with_structured_output(relevance_score_schema)

ranked_docs = []
for doc in docs:
    score = ranking_chain.invoke({"query": enhanced_query, "doc": doc.page_content})["score"]
    ranked_docs.append((doc, float(score)))
    print(f"  Score {score:.1f}: {doc.page_content[:80]}...")

# Sort by score (highest first) and keep top k
ranked_docs.sort(key=lambda x: x[1], reverse=True)
factual_docs = [doc for doc, _ in ranked_docs[:k]]

print(f"\nKept top {len(factual_docs)} documents")

Retrieved 1 documents
  Score 2.0: The Earth is the third planet from the Sun and the only astronomical object know...

Kept top 1 documents


---
## Step 5d: Factual Strategy — Generate the Answer

In [9]:
context = "\n".join([doc.page_content for doc in factual_docs])
factual_answer = answer_chain.invoke({"context": context, "question": query}).content

print(f"Question: {query}")
print(f"\nAnswer: {factual_answer}")

Question: What is the distance between the Earth and the Sun?

Answer: I don't know. The provided text doesn't state the distance between the Earth and the Sun.


---
---
# Demo 2: Analytical Query

**Strategy:** Generate sub-questions → Retrieve for each → Select diverse & relevant docs → Answer

---
## Step 6a: Classify the Query

In [10]:
query = "How does the Earth's distance from the Sun affect its climate?"

category = classify_chain.invoke(query)["category"]
print(f"Query: {query}")
print(f"Category: {category}")

Query: How does the Earth's distance from the Sun affect its climate?
Category: Analytical


---
## Step 6b: Analytical Strategy — Generate Sub-Questions

Complex analytical questions are broken into smaller sub-questions to ensure we cover all angles.

In [11]:
k = 4

sub_queries_prompt = PromptTemplate(
    input_variables=["query", "k"],
    template="Generate {k} sub-questions for: {query}"
)
sub_queries_chain = sub_queries_prompt | llm.with_structured_output(sub_queries_schema)

sub_queries = sub_queries_chain.invoke({"query": query, "k": k})["sub_queries"]

print(f"Original query: {query}")
print(f"\nSub-questions:")
for i, sq in enumerate(sub_queries, 1):
    print(f"  {i}. {sq}")

Original query: How does the Earth's distance from the Sun affect its climate?

Sub-questions:
  1. How does the variation in Earth's distance from the Sun throughout the year (due to its elliptical orbit) impact seasonal temperature differences?
  2. What is the concept of 'Milankovitch cycles,' and how do changes in Earth's orbit (including distance from the Sun) contribute to long-term climate shifts like ice ages?
  3. How does the intensity of solar radiation (affected by distance) influence global average temperatures and precipitation patterns?
  4. Considering the greenhouse effect, how does a slightly closer or farther distance from the Sun impact the overall energy balance of Earth and the resulting climate?


---
## Step 6c: Analytical Strategy — Retrieve for Each Sub-Question

In [12]:
all_docs = []
for sq in sub_queries:
    results = vector_store.similarity_search(sq, k=2)
    all_docs.extend(results)
    print(f"Sub-query: {sq}")
    print(f"  Retrieved {len(results)} docs")

print(f"\nTotal documents retrieved: {len(all_docs)}")

Sub-query: How does the variation in Earth's distance from the Sun throughout the year (due to its elliptical orbit) impact seasonal temperature differences?
  Retrieved 1 docs
Sub-query: What is the concept of 'Milankovitch cycles,' and how do changes in Earth's orbit (including distance from the Sun) contribute to long-term climate shifts like ice ages?
  Retrieved 1 docs
Sub-query: How does the intensity of solar radiation (affected by distance) influence global average temperatures and precipitation patterns?
  Retrieved 1 docs
Sub-query: Considering the greenhouse effect, how does a slightly closer or farther distance from the Sun impact the overall energy balance of Earth and the resulting climate?
  Retrieved 1 docs

Total documents retrieved: 4


---
## Step 6d: Analytical Strategy — Select Diverse & Relevant Documents

With multiple sub-queries, we may have duplicates or overlapping docs. We ask the LLM to pick the most diverse and relevant subset.

In [13]:
diversity_prompt = PromptTemplate(
    input_variables=["query", "docs", "k"],
    template=(
        "Select the most diverse and relevant set of {k} documents for the query: '{query}'\n"
        "Documents: {docs}\n"
        "Return only the indices of selected documents as a list of integers."
    )
)
diversity_chain = diversity_prompt | llm.with_structured_output(selected_indices_schema)

docs_text = "\n".join([f"{i}: {doc.page_content[:50]}..." for i, doc in enumerate(all_docs)])
selected_indices = diversity_chain.invoke({"query": query, "docs": docs_text, "k": k})["indices"]

analytical_docs = [all_docs[i] for i in selected_indices if i < len(all_docs)]

print(f"Selected indices: {selected_indices}")
print(f"Kept {len(analytical_docs)} diverse documents")

Selected indices: []
Kept 0 diverse documents


---
## Step 6e: Analytical Strategy — Generate the Answer

In [14]:
context = "\n".join([doc.page_content for doc in analytical_docs])
analytical_answer = answer_chain.invoke({"context": context, "question": query}).content

print(f"Question: {query}")
print(f"\nAnswer: {analytical_answer}")

Question: How does the Earth's distance from the Sun affect its climate?

Answer: I need the context to answer the question. Please provide the context you're referring to.


---
---
# Demo 3: Opinion Query

**Strategy:** Identify distinct viewpoints → Retrieve for each viewpoint → Select diverse opinions → Answer

---
## Step 7a: Classify the Query

In [15]:
query = "What are the different theories about the origin of life on Earth?"

category = classify_chain.invoke(query)["category"]
print(f"Query: {query}")
print(f"Category: {category}")

Query: What are the different theories about the origin of life on Earth?
Category: Factual


---
## Step 7b: Opinion Strategy — Identify Viewpoints

For opinion queries, we first ask the LLM to identify distinct perspectives on the topic.

In [16]:
k = 3

viewpoints_prompt = PromptTemplate(
    input_variables=["query", "k"],
    template="Identify {k} distinct viewpoints or perspectives on the topic: {query}"
)
viewpoints_chain = viewpoints_prompt | llm

viewpoints_text = viewpoints_chain.invoke({"query": query, "k": k}).content
viewpoints = [v.strip() for v in viewpoints_text.split("\n") if v.strip()]

print(f"Identified viewpoints:")
for i, v in enumerate(viewpoints, 1):
    print(f"  {i}. {v}")

Identified viewpoints:
  1. Okay, let's break down three distinct viewpoints/perspectives on the origin of life on Earth. This is a *massive* and complex topic, and these perspectives represent broad schools of thought, with significant nuance and debate within each. I'll aim for clarity and highlight the core tenets of each.
  2. **1. The "Primordial Soup" (or "Oparin-Haldane") Hypothesis - A Chemical Evolution Perspective (Historically Dominant, Still Influential)**
  3. *   **Core Idea:** Life arose gradually from inorganic matter through a series of chemical reactions in Earth's early oceans.
  4. *   **Key Proponents:** Alexander Oparin (Russian) and J.B.S. Haldane (British) independently proposed this in the 1920s.
  5. *   **Mechanism:**
  6. *   **Early Earth Conditions:**  The early Earth had a reducing atmosphere (rich in gases like methane, ammonia, and hydrogen, with little to no free oxygen).  There was abundant energy from lightning, volcanic activity, and UV radiation.
 

---
## Step 7c: Opinion Strategy — Retrieve for Each Viewpoint

In [17]:
all_docs = []
for viewpoint in viewpoints:
    results = vector_store.similarity_search(f"{query} {viewpoint}", k=2)
    all_docs.extend(results)

print(f"Total documents retrieved across all viewpoints: {len(all_docs)}")

Total documents retrieved across all viewpoints: 38


---
## Step 7d: Opinion Strategy — Select Diverse Opinions

We ask the LLM to pick the most representative and diverse set of documents.

In [18]:
opinion_prompt = PromptTemplate(
    input_variables=["query", "docs", "k"],
    template=(
        "Classify these documents into distinct opinions on '{query}' "
        "and select the {k} most representative and diverse viewpoints:\n"
        "Documents: {docs}\n"
        "Return only the indices of selected documents as a list of integers."
    )
)
opinion_chain = opinion_prompt | llm.with_structured_output(selected_indices_schema)

docs_text = "\n".join([f"{i}: {doc.page_content[:100]}..." for i, doc in enumerate(all_docs)])
selected_indices = opinion_chain.invoke({"query": query, "docs": docs_text, "k": k})["indices"]

opinion_docs = [all_docs[i] for i in selected_indices if i < len(all_docs)]

print(f"Selected indices: {selected_indices}")
print(f"Kept {len(opinion_docs)} diverse opinion documents")

Selected indices: []
Kept 0 diverse opinion documents


---
## Step 7e: Opinion Strategy — Generate the Answer

In [19]:
context = "\n".join([doc.page_content for doc in opinion_docs])
opinion_answer = answer_chain.invoke({"context": context, "question": query}).content

print(f"Question: {query}")
print(f"\nAnswer: {opinion_answer}")

Question: What are the different theories about the origin of life on Earth?

Answer: You haven't provided any context. I need the context to answer the question "What are the different theories about the origin of life on Earth?". Please provide the text you want me to use.



---
---
# Demo 4: Contextual Query

**Strategy:** Incorporate user context → Retrieve with contextualized query → Rank considering context → Answer

---
## Step 8a: Classify the Query

In [20]:
query = "How does the Earth's position in the Solar System influence its habitability?"

category = classify_chain.invoke(query)["category"]
print(f"Query: {query}")
print(f"Category: {category}")

Query: How does the Earth's position in the Solar System influence its habitability?
Category: Analytical


---
## Step 8b: Contextual Strategy — Incorporate User Context

Contextual queries depend on *who* is asking. We provide user context and ask the LLM to reformulate the query accordingly.

In [21]:
user_context = "The user is an astronomy student studying planetary science."

context_prompt = PromptTemplate(
    input_variables=["query", "context"],
    template="Given the user context: {context}\nReformulate the query to best address the user's needs: {query}"
)
context_chain = context_prompt | llm

contextualized_query = context_chain.invoke({"query": query, "context": user_context}).content

print(f"Original query: {query}")
print(f"User context: {user_context}")
print(f"\nContextualized query: {contextualized_query}")

Original query: How does the Earth's position in the Solar System influence its habitability?
User context: The user is an astronomy student studying planetary science.

Contextualized query: Okay, here are a few reformulated queries, tailored for an astronomy/planetary science student, ranging from broad to more specific. I've included explanations of why each is better than the original.

**1. (Broad & Exploratory - Good starting point)**

* **Query:** "What are the key astronomical factors determining Earth's habitability, and how do they relate to our Solar System's architecture?"
* **Why it's better:** This expands beyond just "position" to include other relevant astronomical factors (like the Sun's properties, orbital stability, etc.).  The phrase "Solar System's architecture" encourages a discussion of the overall system design and how that contributes.  It's open-ended enough to allow for a comprehensive answer.

**2. (More Focused - Good for a specific assignment)**

* **Query

---
## Step 8c: Contextual Strategy — Retrieve and Rank with Context

We retrieve with the contextualized query, then rank each document considering both relevance and user context.

In [22]:
k = 4

# Retrieve 2x documents
docs = vector_store.similarity_search(contextualized_query, k=k * 2)
print(f"Retrieved {len(docs)} documents")

# Rank by relevance considering user context
ctx_ranking_prompt = PromptTemplate(
    input_variables=["query", "context", "doc"],
    template=(
        "Given the query: '{query}' and user context: '{context}', "
        "rate the relevance of this document on a scale of 1-10:\n"
        "Document: {doc}\nRelevance score:"
    )
)
ctx_ranking_chain = ctx_ranking_prompt | llm.with_structured_output(relevance_score_schema)

ranked_docs = []
for doc in docs:
    score = ctx_ranking_chain.invoke({
        "query": contextualized_query,
        "context": user_context,
        "doc": doc.page_content
    })["score"]
    ranked_docs.append((doc, float(score)))
    print(f"  Score {score:.1f}: {doc.page_content[:80]}...")

# Sort and keep top k
ranked_docs.sort(key=lambda x: x[1], reverse=True)
contextual_docs = [doc for doc, _ in ranked_docs[:k]]

print(f"\nKept top {len(contextual_docs)} documents")

Retrieved 1 documents
  Score 3.0: The Earth is the third planet from the Sun and the only astronomical object know...

Kept top 1 documents


---
## Step 8d: Contextual Strategy — Generate the Answer

In [23]:
context = "\n".join([doc.page_content for doc in contextual_docs])
contextual_answer = answer_chain.invoke({"context": context, "question": query}).content

print(f"Question: {query}")
print(f"User context: {user_context}")
print(f"\nAnswer: {contextual_answer}")

Question: How does the Earth's position in the Solar System influence its habitability?
User context: The user is an astronomy student studying planetary science.

Answer: I don't know. The provided text only states that Earth is the third planet from the Sun and harbors life, but doesn't explain *how* its position influences habitability.


---
---
## Summary

| Strategy | When to use | Key idea |
|---|---|---|
| **Factual** | Specific, verifiable questions | Enhance query → rank by relevance |
| **Analytical** | Complex, multi-faceted questions | Break into sub-questions → ensure diversity |
| **Opinion** | Subjective / multiple-viewpoint topics | Identify viewpoints → retrieve per viewpoint |
| **Contextual** | User-dependent questions | Reformulate with user context → rank with context |

The adaptive approach ensures each query gets the retrieval strategy that suits it best, rather than one-size-fits-all.